In [1]:
import fine.IOManagement.xarrayIO as xrIO

%load_ext autoreload
%autoreload 2

# How to save an energy system model instance and set it back up? 

**Xarray and NetCDF files to the rescue!** The data contained within an Energy System Model (ESM) instance and the optimization results is vast and complex. Saving it directly is not possible. It can, however, be saved as a NetCDF file which supports complex data structures. 

#### What exactly is NetCDF? 
NetCDF (Network Common Data Format) is a set of software libraries and machine-independent data formats that support the creation, access, and sharing of array-oriented scientific data. It is also a community standard for sharing scientific data. 

#### Python modules that support working with NetCDF files:
1. netcdf4-python: Official Python interface to netCDF files
2. PyNIO: To access different file formats such as netCDF, HDF, and GRIB
3. xarray: Based on NumPy and pandas

Note: xarray module is used here. 

For our use case, the following functionalities are provided: 
* Conversion of ESM instance to xarray dataset. Additionally, possible to save this dataset as NetCDF file in a desired folder, with a desired file name. 
* Conversion of xarray dataset/saved NetCDF file back to ESM instance.

#### High-level structure of the data: 

<img src="overall_structure.png" style="width: 1000px;"/>


#### Structure of xarray dataset - For a non-transmission component: 

<img src="non_transmission.png" style="width: 1000px;"/>

#### Structure of xarray dataset - For a transmission component: 

<img src="transmission.png" style="width: 1000px;"/>


## Conversion of ESM instance to xarray dataset and saving it as a NetCDF file

### STEP 1. Set up your  ESM instance 

In [2]:
from getModel import getModel

esM = getModel()
esM.optimize()

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpooy9exlp.glpk.raw --wglp /tmp/tmpe9wwwlau.glpk.glp --cpxlp
 /tmp/tmp2szftz9q.pyomo.lp
Reading problem data from '/tmp/tmp2szftz9q.pyomo.lp'...
96 rows, 74 columns, 240 non-zeros
621 lines were read
Writing problem data to '/tmp/tmpe9wwwlau.glpk.glp'...
479 lines were written
GLPK Simplex Optimizer 5.0
96 rows, 74 columns, 240 non-zeros
Preprocessing...
66 rows, 43 columns, 176 non-zeros
Scaling...
 A: min|aij| =  3.300e-01  max|aij| =  2.190e+03  ratio =  6.636e+03
GM: min|aij| =  2.277e-01  max|aij| =  4.391e+00  ratio =  1.928e+01
EQ: min|aij| =  5.216e-02  max|aij| =  1.000e+00  ratio =  1.917e+01
Constructing initial basis...
Size of triangular part is 63
      0: obj =   3.585342857e+06 inf =   6.052e+06 (8)
      6: obj =   4.331272582e+06 inf =   1.746e-10 (0)
*    20: obj =   3.883295266e+06 inf =   2.910e-11 (0)
OPTIMAL LP SOLUTION FOUND
Time used:   0.0 secs
Memory used: 0.1 Mb (119919

### STEP 2. Conversion to xarray datasets and saving as NetCDF file
You can convert the esM to xarray datasets with `esm_to_datasets` and access Input, Parameters or Result.


In [3]:
esm_datasets = xrIO.writeEnergySystemModelToDatasets(esM)

In [4]:
esm_datasets["Input"]["Sink"]["Industry site"][
    "ts_operationRateFix"
].to_dataframe().unstack()

ts_operationRateFix                 
space ElectrolyzerLocation IndustryLocation
time                                       
0                      0.0       13140000.0
1                      0.0       13140000.0
2                      0.0       13140000.0
3                      0.0       13140000.0

In [5]:
esm_datasets["Results"][0]["SourceSinkModel"]["Electricity market"]

<xarray.Dataset> Size: 544B
Dimensions:                          (space: 2, time: 4)
Coordinates:
  * time                             (time) int64 32B 0 1 2 3
  * space                            (space) <U20 160B 'ElectrolyzerLocation'...
Data variables: (12/19)
    NPVcontribution                  (space) float64 16B 1.52e+06 0.0
    TAC                              (space) float64 16B 1.52e+06 0.0
    annual_operation                 (space) float64 16B 7.509e+07 nan
    capacity                         (space) float64 16B nan nan
    capexCap                         (space) float64 16B nan nan
    capexIfBuilt                     (space) float64 16B nan nan
    ...                               ...
    opexCap                          (space) float64 16B nan nan
    opexIfBuilt                      (space) float64 16B nan nan
    opexOp                           (space) float64 16B 0.0 nan
    revenueLifetimeShorteningResale  (space) float64 16B nan nan
    total_operation                  (space) float64 16B 7.509e+07 nan
    operationVariablesOptimum        (time, space) float64 64B 1.877e+07 ... nan
Attributes: (12/18)
    NPVcontribution:                  [1 Euro]
    TAC:                              [1 Euro/a]
    annual_operation:                 [kW$_{el}$*h/a]
    capacity:                         [kW$_{el}$]
    capexCap:                         [1 Euro/a]
    capexIfBuilt:                     [1 Euro/a]
    ...                               ...
    isBuilt:                          [-]
    opexCap:                          [1 Euro/a]
    opexIfBuilt:                      [1 Euro/a]
    opexOp:                           [1 Euro/a]
    revenueLifetimeShorteningResale:  [1 Euro]
    total_operation:                  [kW$_{el}$*h]

In [6]:
esm_datasets["Parameters"]

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
Attributes: (12/15)
    locations:                  {'IndustryLocation', 'ElectrolyzerLocation'}
    commodities:                {'hydrogen', 'electricity'}
    commodityUnitsDict:         {'electricity': 'kW$_{el}$', 'hydrogen': 'kW$...
    numberOfTimeSteps:          4
    hoursPerTimeStep:           2190
    startYear:                  0
    ...                         ...
    costUnit:                   1 Euro
    lengthUnit:                 km
    verboseLogLevel:            1
    balanceLimit:               None
    pathwayBalanceLimit:        None
    annuityPerpetuity:          False

Or save it directly to NetCDF with `esm_to_netcdf`:

In [7]:
_ = xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath="my_esm.nc", overwriteExisting=True
)

### STEP 3. Load esM from NetCDF file or xarray datasets

You can load an esM from file with `netcdf_to_esm`.

In [8]:
esm_from_netcdf = xrIO.readNetCDFtoEnergySystemModel("my_esm.nc")

In [9]:
esm_from_netcdf.getComponentAttribute("Industry site", "operationRateFix")

space,ElectrolyzerLocation,IndustryLocation
time,,
0,0.0,13140000.0
1,0.0,13140000.0
2,0.0,13140000.0
3,0.0,13140000.0


Or from datasets with `datasets_to_esm`.

In [10]:
esm_from_datasets = xrIO.convertDatasetsToEnergySystemModel(esm_datasets)

In [11]:
esm_from_datasets.getComponentAttribute("Industry site", "operationRateFix")

space,ElectrolyzerLocation,IndustryLocation
time,,
0,0.0,13140000.0
1,0.0,13140000.0
2,0.0,13140000.0
3,0.0,13140000.0


In [12]:
esm_datasets["Results"][0]["SourceSinkModel"]["Electricity market"][
    "operationVariablesOptimum"
]

<xarray.DataArray 'operationVariablesOptimum' (time: 4, space: 2)> Size: 64B
array([[18771428.5714286,              nan],
       [37542857.1428571,              nan],
       [       0.       ,              nan],
       [18771428.5714286,              nan]])
Coordinates:
  * time     (time) int64 32B 0 1 2 3
  * space    (space) <U20 160B 'ElectrolyzerLocation' 'IndustryLocation'